# Wynr-1.0 — MoE Kaggle Notebook

**Mục tiêu**: smoke + train mô hình Wynr-1.0 (decoder-only Transformer + MoE) trên Kaggle GPU miễn phí.

**Cảnh báo giới hạn tài nguyên**:
- GPU miễn phí thường là Tesla T4 (hoặc GPU khác tùy phiên), VRAM giới hạn.
- `/kaggle/working` có thể mất khi runtime kết thúc — không nên giữ trạng thái quan trọng chỉ ở đây.
- Không tự động tải pretrained weights; Wynr-1.0 train từ random init.
- Không chạy một cell duy nhất quá lâu; có log tiến độ từng bước.

In [ ]:
# 2. Kiểm tra Python, PyTorch, CUDA, GPU và VRAM
import sys, os, json, random, time, math
from pathlib import Path
import torch
print('python', sys.version.split()[0])
print('torch', torch.__version__)
print('cuda available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))
    print('vram MB', torch.cuda.get_device_properties(0).total_memory / 1024 / 1024)
else:
    print('NO CUDA - se chay tren CPU')

In [ ]:
# 3. Cài dependency toi nho
!pip install -q torch --index-url https://download.pytorch.org/whl/cpu
!pip install -q sentencepiece pyyaml numpy

In [ ]:
# 4. Dat seed
torch.manual_seed(42)
random.seed(42)
import numpy as np
np.random.seed(42)
print('seed ok')

In [ ]:
# 5. Gan ho ho doc dataset dau vao
DATA_DIR = Path('/kaggle/working/data')
if (DATA_DIR / 'train' / 'train.jsonl').exists():
    print('dataset found')
else:
    print('no dataset yet - se tao tai day o cell 8')

In [ ]:
# 6. Khoi phuc checkpoint va tokenizer tu artifact da luu
CKPT = Path('/kaggle/working/checkpoints/wynr_step00006_best.ckpt')
TOKENIZER_DIR = Path('/kaggle/working/checkpoints/tokenizer/wynr-1.0')
print('checkpoint exists', CKPT.exists())
print('tokenizer exists', TOKENIZER_DIR.exists())

In [ ]:
# 7. Kiem tra manifest, checksum va tuong thich
import hashlib, json as _json
manifest_path = Path('/kaggle/working/artifacts/manifest.json')
if manifest_path.exists():
    manifest = _json.loads(manifest_path.read_text())
    print('manifest status', manifest.get('status'))
print('verify:')
!python scripts/verify_checkpoint.py --checkpoint /kaggle/working/checkpoints/wynr_step00006_best.ckpt --config /kaggle/working/configs/smoke.yaml

In [ ]:
# 8. Tao du lieu mau neu chua co du lieu
!python scripts/generate_data.py --count 120 --output /kaggle/working/data/train/train.jsonl
!python scripts/validate_data.py --input /kaggle/working/data/train/train.jsonl --stats-output /kaggle/working/data/stats.json
!python scripts/deduplicate_data.py --input /kaggle/working/data/train/train.jsonl --output /kaggle/working/data/processed/train.jsonl

In [ ]:
# 9. Kiem tra va lo bo (stats da duoc in o cell 8)
import json as _json
stats = _json.loads(Path('/kaggle/working/data/stats.json').read_text())
print('num_samples', stats.get('num_samples'), 'num_clean', stats.get('num_clean'), 'dup_removed', stats.get('num_duplicates_removed'))

In [ ]:
# 10. Train/validation split
from src.wynr.data_utils import split_train_val
train_texts = [l for l in Path('/kaggle/working/data/train/train.jsonl').read_text(encoding='utf-8').splitlines() if l.strip()]
import json as _json2
train_texts = [_json2.loads(l)['text'] for l in train_texts]
train_split, val_split = split_train_val(train_texts, train_ratio=0.9, seed=42)
Path('/kaggle/working/data/train_split.jsonl').write_text('\n'.join(train_split), encoding='utf-8')
Path('/kaggle/working/data/val_split.jsonl').write_text('\n'.join(val_split), encoding='utf-8')
print('train split', len(train_split), 'val split', len(val_split))

In [ ]:
# 11. Huan luyen tokenizer neu chua co phien ban tuong thich
from src.wynr.tokenizer import WynrTokenizer
if not TOKENIZER_DIR.exists():
    tok = WynrTokenizer.train(texts=train_texts, model_dir=TOKENIZER_DIR, source='synthetic_v1')
    print('tokenizer trained')
else:
    tok = WynrTokenizer.load(TOKENIZER_DIR)
    print('tokenizer loaded')

In [ ]:
# 12. Tokenize du lieu
def tok_encode(texts, max_len=64):
    seqs = []
    for t in texts:
        ids = [i for i in tok.encode(t) if i != tok.pad_id]
        seqs.append(ids[:max_len])
    return seqs

train_seqs = tok_encode(train_texts)
print('num tokenized', len(train_seqs), 'avg len', sum(len(s) for s in train_seqs)//max(len(train_seqs),1))

In [ ]:
# 13. Khoi tao mo hinh tu trong so ngau nhien neu chua co checkpoint
from src.wynr.config import ModelConfig, load_config
from src.wynr.model import WynrMoE
cfg = load_config('/kaggle/working/configs/smoke.yaml')
model = WynrMoE(ModelConfig(**vars(cfg.model)))
if CKPT.exists():
    from src.wynr.checkpoint import load_checkpoint
    load_checkpoint(str(CKPT), model)
    print('loaded checkpoint weights')
print('model ready (random init if no checkpoint)')

In [ ]:
# 14. Kiem so tham so
print('model param count', model.count_parameters())
print('config', vars(cfg.model))

In [ ]:
# 15. Kiem bo nho bang batch nho
batch = torch.tensor([train_seqs[0][:64]], dtype=torch.long)
logits, aux = model(batch, return_aux=True)
print('small batch forward ok', tuple(logits.shape), float(aux))

In [ ]:
# 16. Chay smoke test
from src.wynr.training import Trainer
import torch
trainer = Trainer(model=model, optimizer=torch.optim.AdamW(model.parameters(), lr=3e-4), scheduler=None, scaler=None, config=cfg, device='cuda' if torch.cuda.is_available() else 'cpu')
loss_info = trainer.train_step(batch)
print('smoke train step loss', loss_info)

In [ ]:
# 17. Huan luyen chinh
!python scripts/train.py --config /kaggle/working/configs/smoke.yaml --mode train

In [ ]:
# 18. Validation dinh ky
val_seq = train_seqs[len(train_seqs)//2]
val_info = trainer.validate_step(torch.tensor([val_seq], dtype=torch.long))
print('val info', val_info)

In [ ]:
# 19. Luu checkpoint
from src.wynr.checkpoint import save_checkpoint
save_checkpoint(path='/kaggle/working/checkpoints/wynr_step00099.ckpt', model=model, optimizer=torch.optim.AdamW(model.parameters(), lr=3e-4), scheduler=None, scaler=None, global_step=99, best_validation_loss=None, model_config=vars(cfg.model), training_config=vars(cfg.training), tokenizer_info={'source': 'synthetic_v1', 'vocab_size': cfg.tokenizer.vocab_size}, source='kaggle_notebook')

In [ ]:
# 20. Hien thi loss
import json as _json3
history = [json.loads(l) for l in Path('/kaggle/working/logs/training_history.jsonl').read_text().splitlines() if l.strip()]
print('last history', history[-1] if history else None)

In [ ]:
# 21. Sinh van ban thu nghiem
from src.wynr.generation import generate
gen_text, _ = generate(model=model, tokenizer=tok, prompt='Xin chao', max_new_tokens=20, temperature=1.0, top_k=5, top_p=0.9)
print('generated:', gen_text)

In [ ]:
# 22. Kiem tra checkpoint
!python scripts/verify_checkpoint.py --checkpoint /kaggle/working/checkpoints/wynr_step00099.ckpt --config /kaggle/working/configs/smoke.yaml

In [ ]:
# 23. Dong goi artifact
!python scripts/package_artifacts.py --input-dir /kaggle/working/artifacts --output-zip /kaggle/working/artifacts/wynr_artifacts.zip --dry-run

## 24. Huong dan xuat artifact khoi runtime

- Sau khi train xong, chay: `python scripts/export_kaggle_artifacts.py --checkpoint /kaggle/working/checkpoints/wynr_step000XX.ckpt --output-dir /kaggle/working/artifacts_export`
- Tai thu muc `/kaggle/working/artifacts_export` ra ngoai (download output).
- Ho hoac tao Kaggle Dataset moi: UI Kaggle -> Datasets -> New Dataset -> upload artifact.

## 25. Huong dan tao phien ban dataset moi

- Kaggle: Datasets -> Create new dataset -> upload artifact ZIP hoac thu muc checkpoint/tokenizer.
- Can quyen write cua tai khoan; khong tu dong upload khi chua cau hinh token/API key.
- Sau do trong notebook moi: attach dataset phien moi vao kernel.

## 26. Huong dan bat dau phien tiep theo va resume

```
python scripts/train.py --config configs/smoke.yaml --mode resume --checkpoint /kaggle/working/checkpoints/wynr_step000XX.ckpt
python scripts/generate.py --checkpoint /kaggle/working/checkpoints/wynr_step000XX_best.ckpt --prompt "Xin chao" --max-new-tokens 64
```

In [ ]:
# 27. Tao bao cao trang thai phien
session_report = {
    'session': 'kaggle_notebook',
    'timestamp': time.time(),
    'status': 'TRAIN COMPLETED' if not CKPT.exists() else 'CHECKPOINT PRESENT',
    'checkpoint': str(CKPT),
    'tokenizer': str(TOKENIZER_DIR),
}
Path('/kaggle/working/session_report.json').write_text(json.dumps(session_report, indent=2), encoding='utf-8')
print('session report written')